In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = os.getenv("CLAUDE_MODEL")

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [9]:
import json

# For step 2, where we create an evaluation dataset. We ask Claude to do it for us.
def generate_dataset():
    prompt = """
		Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
		that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
		each representing task that requires Python, JSON, or a Regex to complete.

		Example output:
		```json
		[
			{
				"task": "Description of task",
                "format": "json" or "python" or "regex"
			},
			...additional
		]
		```

		* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
		* Focus on tasks that do not require writing much code

		Please generate 3 objects.
	"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [13]:
dataset = generate_dataset()

with open("dataset.json", "w") as file:
  json.dump(dataset, file, indent=2)

dataset

[{'task': 'Create a JSON configuration object for an AWS Lambda function that processes S3 events, with environment variables for the bucket name and log level',
  'format': 'json'},
 {'task': "Write a Python function that takes an AWS IAM policy JSON string and returns a list of all the AWS service actions (e.g., 's3:GetObject') granted in that policy",
  'format': 'python'},
 {'task': 'Create a regular expression that validates AWS S3 bucket names according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens, must start and end with letter or number)',
  'format': 'regex'}]

In [10]:
def run_prompt(test_case):
	"""Merges the prompt and test case input, then returns the result"""

	prompt = f"""
		Please solve the following task:
		{test_case["task"]}

		* Respond only with Python, Json, or a plain Regex
		* Do not add any comments or commentary or explanation
	"""

	messages = []
	add_user_message(messages, prompt)
	add_assistant_message(messages, "```code") # we could put ```python or ```regex or ```json, but ```code covers all 3
	output = chat(messages, stop_sequences=["```"])
	return output

In [11]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
		You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

		Original Task:
		<task>
		{test_case["task"]}
		</task>

		Solution to Evaluate:
		<solution>
		{output}
		</solution>

		Output Format
		Provide your evaluation as a structured JSON object with the following fields, in this specific order:
		- "strengths": An array of 1-3 key strengths
		- "weaknesses": An array of 1-3 key areas for improvement
		- "reasoning": A concise explanation of your overall assessment
		- "score": A number between 1-10

		Respond with JSON. Keep your response concise and direct.
		Example response shape:
		{{
			"strengths": string[],
			"weaknesses": string[],
			"reasoning": string,
			"score": number
		}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)
  

In [12]:
import re
import ast

def validate_json(text):
	try:
		json.loads(text.strip())
		return 10
	except json.JSONDecodeError:
		return 0
	
def validate_python(text):
	try:
		ast.parse(text.strip())
		return 10
	except SyntaxError:
		return 0
	
def validate_regex(text):
	try:
		re.compile(text.strip())
		return 10
	except re.error:
		return 0
	
def grade_syntax(response, test_case):
	format = test_case["format"]
	if format == "json":
		return validate_json(response)
	elif format == "python":
		return validate_python(response)
	elif format == "regex":
		return validate_regex(response)
	else:
		return -1

In [13]:
def run_test_case(test_case):
	"""Calls run_prompt, then grades the result"""
	output = run_prompt(test_case)

	# model grading
	model_grade = grade_by_model(test_case, output)
	model_score = model_grade["score"]
	reasoning = model_grade["reasoning"]

	# code grading
	syntax_score = grade_syntax(output, test_case)

	score = (model_score + syntax_score) / 2

	return {
		"output" : output,
		"test_case" : test_case,
		"score" : score,
		"reasoning" : reasoning
	}


In [14]:
from statistics import mean

def run_eval(dataset):
	"""Loads the dataset and calls run_test_case with each case"""
	results = []

	for test_case in dataset:
		result = run_test_case(test_case)
		results.append(result)

	average_score = mean([result["score"] for result in results])
	print(f"Average score: {average_score}")
	return results

In [15]:
with open("dataset.json", "r") as file:
	dataset = json.load(file)

results = run_eval(dataset)

Average score: 7.666666666666667


In [16]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n  \"FunctionName\": \"S3EventProcessor\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::ACCOUNT_ID:role/lambda-s3-execution-role\",\n  \"Handler\": \"index.handler\",\n  \"Timeout\": 60,\n  \"MemorySize\": 256,\n  \"Environment\": {\n    \"Variables\": {\n      \"BUCKET_NAME\": \"my-s3-bucket\",\n      \"LOG_LEVEL\": \"INFO\"\n    }\n  },\n  \"Events\": {\n    \"S3Event\": {\n      \"Type\": \"S3\",\n      \"Properties\": {\n        \"Bucket\": \"my-s3-bucket\",\n        \"Events\": [\n          \"s3:ObjectCreated:*\",\n          \"s3:ObjectRemoved:*\"\n        ]\n      }\n    }\n  }\n}\n",
    "test_case": {
      "task": "Create a JSON configuration object for an AWS Lambda function that processes S3 events, with environment variables for the bucket name and log level",
      "format": "json"
    },
    "score": 7.5,
    "reasoning": "The solution correctly addresses the core requirement of including environment variables for bucket name and lo